# 07 — Research Module: Scientific Productivity Forecast
**EduForecast** | OpenAlex API | Notebook 7

Forecasts yearly publication counts for Astana IT University using the public OpenAlex API, comparing Prophet `growth='linear'` vs `growth='logistic'`.

**Scope limitation (read before trusting any number below):** this fits on **6-7 yearly data points** (2020-2026; 2019 excluded, see cell below). That is far too little data for a high-precision forecast — confidence intervals are wide by construction, and the only validation performed is a single leave-last-year-out sanity check, not cross-validation. Treat this as a proof-of-concept for the modelling pipeline, not a production-grade forecast.

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.research.aggregate import build_yearly_counts, filter_training_window, to_prophet_frame
from src.research.config import (
    FIGURES_DIR, METRICS_DIR, OPENALEX_INSTITUTION_NAME,
    TRAINING_YEAR_MIN, TRAINING_YEAR_MAX, LOGISTIC_CAP_MULTIPLIER,
)
from src.research.forecast import run_forecast_comparison

print(f'Institution: {OPENALEX_INSTITUTION_NAME}')
print(f'Training window: {TRAINING_YEAR_MIN}-{TRAINING_YEAR_MAX}')

Institution: Astana IT University
Training window: 2020-2026


## 1. Collect & Aggregate (OpenAlex)

In [2]:
full_counts = build_yearly_counts()
full_counts

,year,works_count
0,2019,1
1,2020,26
2,2021,83
3,2022,156
4,2023,252
5,2024,416
6,2025,606
7,2026,753


In [3]:
# 2019 is excluded: a single indexed publication that year reflects the
# institution's pre-registration/startup period, not an actual annual
# publication rate. Including it would distort both the trend fit and
# the logistic cap computed below.
training = filter_training_window(full_counts)
training

,year,works_count
0,2020,26
1,2021,83
2,2022,156
3,2023,252
4,2024,416
5,2025,606
6,2026,753


## 2. Forecast: linear vs logistic growth

The logistic variant needs a saturating `cap`. There is no way to derive this from 7 points, so it is set explicitly as an **expert guess** (current peak × 2, see `src/research/config.py::LOGISTIC_CAP_MULTIPLIER`), not a value fit from the data.

In [4]:
comparison = run_forecast_comparison(training)
print(f'Logistic cap (expert guess): {comparison.logistic_cap:.0f}')
comparison.linear_forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(4)

16:29:15 - cmdstanpy - INFO - Chain [1] start processing


16:29:15 - cmdstanpy - INFO - Chain [1] done processing


16:29:15 - cmdstanpy - INFO - Chain [1] start processing


16:29:15 - cmdstanpy - INFO - Chain [1] done processing


16:29:15 - cmdstanpy - INFO - Chain [1] start processing


16:29:15 - cmdstanpy - INFO - Chain [1] done processing


16:29:15 - cmdstanpy - INFO - Chain [1] start processing


16:29:15 - cmdstanpy - INFO - Chain [1] done processing


Logistic cap (expert guess): 1506


,ds,yhat,yhat_lower,yhat_upper
6,2026-01-01,700.949404,639.520839,766.907784
7,2027-01-01,825.358765,756.553073,891.858743
8,2028-01-01,949.768126,886.364427,1010.442401
9,2029-01-01,1074.518335,1010.344044,1141.490795


In [5]:
comparison.logistic_forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(4)

,ds,yhat,yhat_lower,yhat_upper
6,2026-01-01,782.445674,754.796858,809.329567
7,2027-01-01,982.801030,953.616842,1009.344331
8,2028-01-01,1152.725990,1126.127233,1180.060885
9,2029-01-01,1280.435563,1254.233843,1307.211751


In [6]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(training['year'], training['works_count'], 'o-', color='#0f172a', label='Факт')

last_year = training['year'].max()
for fcst, label, color in (
    (comparison.linear_forecast, 'Прогноз (linear)', '#2563eb'),
    (comparison.logistic_forecast, 'Прогноз (logistic)', '#059669'),
):
    future = fcst[fcst['ds'].dt.year > last_year].copy()
    future['year'] = future['ds'].dt.year
    ax.plot(future['year'], future['yhat'], '--o', color=color, label=label)
    ax.fill_between(future['year'], future['yhat_lower'], future['yhat_upper'], color=color, alpha=0.15)

ax.axhline(comparison.logistic_cap, color='#6b7280', linestyle=':', linewidth=1,
           label=f'Logistic cap (expert guess) = {comparison.logistic_cap:.0f}')
ax.set_xlabel('Год'); ax.set_ylabel('Публикаций в год')
ax.set_title(f'{OPENALEX_INSTITUTION_NAME}: прогноз научной продуктивности')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / '07_forecast_comparison.png', dpi=150)
plt.show()

## 3. Backtest (sanity check, NOT cross-validation)

Train on 2020-2025, predict 2026, compare against the actual observed count (753). With only 6-7 yearly points there is not enough data for a statistically meaningful k-fold or time-series cross-validation — this is a single held-out-year sanity check, named as such on purpose.

In [7]:
bt = comparison.backtest
print(f'Backtest year: {bt.backtest_year}')
print(f'Actual: {bt.actual:.0f}')
print()
for growth, res in bt.predictions.items():
    print(f"{growth:10s} predicted={res['predicted']:6.0f}  "
          f"[{res['lower']:.0f}, {res['upper']:.0f}]  "
          f"abs_error={res['abs_error']:6.1f}  pct_error={res['pct_error']:5.1f}%")
print()
print(bt.note)

Backtest year: 2026
Actual: 753

linear     predicted=   656  [599, 712]  abs_error=  97.1  pct_error= 12.9%
logistic   predicted=   801  [790, 811]  abs_error=  48.2  pct_error=  6.4%

Leave-last-year-out sanity check on a single held-out point. NOT k-fold or time-series cross-validation: with only 6-7 yearly observations there isn't enough data for a statistically meaningful CV.


In [8]:
backtest_df = pd.DataFrame([
    {'growth': g, 'backtest_year': bt.backtest_year, 'actual': bt.actual, **res}
    for g, res in bt.predictions.items()
])
backtest_df.to_csv(METRICS_DIR / 'research_backtest.csv', index=False)
backtest_df

,growth,backtest_year,actual,predicted,lower,upper,abs_error,pct_error
0,linear,2026,753.0,655.869145,598.734755,712.146942,97.130855,12.899184
1,logistic,2026,753.0,801.195212,790.356437,811.175214,48.195212,6.400427


## 4. Summary

- Data source: OpenAlex API, institution ID `I4405257690` (Astana IT University), aggregated at the whole-institution level (no per-department breakdown — OpenAlex doesn't expose clean sub-institution IDs; see PROJECT_STATUS.md §4.4).
- Training window: 2020-2026 (7 points), 2019 dropped as a non-representative startup-period outlier.
- Two Prophet variants compared: `linear` (no ceiling) and `logistic` (ceiling = expert-guessed cap, not data-derived).
- Validation: one leave-last-year-out sanity check, explicitly not a cross-validation, given the small N.
- **This is a proof-of-concept.** Revisit the logistic cap and re-run the backtest once a few more years of data are available.